# NB4 · Demo reti neurali

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/battles5/conad-statistical-learning/blob/main/notebooks/NB4_reti_neurali.ipynb)

**una piccola rete che richiama il mattino: e' ancora apprendimento statistico**

corso *Apprendimento statistico* per CONAD Nord Ovest · Orso Peruzzi (IFAB)

> come si esegue una cella: clicca dentro e premi **Shift + Invio**; esegui le celle in ordine, dall'alto verso il basso.

## 1. una rete neurale e' ancora statistical learning

usiamo gli stessi dati *Default* e una piccola rete (un MLP, *multi-layer perceptron*). valgono gli stessi concetti del mattino: training/test, overfitting, regolarizzazione. e' lo spirito del capitolo 10 di ISL.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def carica(nome):
    "carica un csv dal repo GitHub (Colab) o dalla cartella locale ../data (fallback)"
    base = "https://raw.githubusercontent.com/battles5/conad-statistical-learning/main/data/"
    try:
        return pd.read_csv(base + nome)
    except Exception:
        return pd.read_csv("../data/" + nome)

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

df = carica("Default.csv")
df["default"] = (df["default"] == "Yes").astype(int)
df["student"] = (df["student"] == "Yes").astype(int)
X = df[["balance", "income", "student"]].values
y = df["default"].values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.30, random_state=0, stratify=y)

## 2. una piccola rete

due strati nascosti. la "manopola" qui e' `ALPHA`, la forza della regolarizzazione (come il lambda di prima).

> **manopola**: prova `ALPHA = 0.0001` (poca regolarizzazione) e `ALPHA = 1.0` (tanta). guarda la differenza tra AUC di training e di test.

In [3]:
from sklearn.neural_network import MLPClassifier

# >>> MANOPOLA: regolarizzazione della rete <<<
ALPHA = 0.0001

rete = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(32, 16), alpha=ALPHA, max_iter=400, random_state=0),
).fit(Xtr, ytr)

auc_tr = roc_auc_score(ytr, rete.predict_proba(Xtr)[:, 1])
auc_te = roc_auc_score(yte, rete.predict_proba(Xte)[:, 1])
print(f"alpha = {ALPHA}")
print(f"AUC training: {auc_tr:.3f}   AUC test: {auc_te:.3f}")
print("differenza (overfitting se grande):", round(auc_tr - auc_te, 3))

alpha = 0.0001
AUC training: 0.945   AUC test: 0.959
differenza (overfitting se grande): -0.014


## 3. confronto con la baseline lineare

la rete batte davvero la semplice logistica? su questi dati spesso no: piu' flessibilita' non e' sempre meglio.

In [4]:
from sklearn.linear_model import LogisticRegression

logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
print("AUC test logistica:", round(roc_auc_score(yte, logit.predict_proba(Xte)[:, 1]), 3))
print("AUC test rete:     ", round(auc_te, 3))

AUC test logistica: 0.961
AUC test rete:      0.959


stesso identico schema del mattino: train/test, overfitting, regolarizzazione. la rete e' solo una `f` piu' flessibile, non un mondo a parte.

---

### cella bonus: la stessa rete in Keras (TensorFlow)

Colab ha gia' TensorFlow installato. questa cella mostra la stessa idea con Keras; se esegui in locale senza TensorFlow, stampa solo un avviso.

In [5]:
# BONUS (gira su Colab, dove TensorFlow e' preinstallato)
try:
    from tensorflow import keras

    scaler = StandardScaler().fit(Xtr)
    modello = keras.Sequential([
        keras.layers.Input(shape=(3,)),
        keras.layers.Dense(32, activation="relu"),
        keras.layers.Dense(16, activation="relu"),
        keras.layers.Dense(1, activation="sigmoid"),
    ])
    modello.compile(optimizer="adam", loss="binary_crossentropy", metrics=["AUC"])
    modello.fit(scaler.transform(Xtr), ytr, epochs=10, batch_size=64, verbose=0)
    auc = modello.evaluate(scaler.transform(Xte), yte, verbose=0)[1]
    print("AUC test (Keras):", round(float(auc), 3))
except ImportError:
    print("TensorFlow non disponibile qui: esegui questa cella su Colab.")

TensorFlow non disponibile qui: esegui questa cella su Colab.


## 4. la big picture

la stessa idea, scalata, porta a:

- **CNN** per le immagini;
- **RNN** e **LSTM** per le sequenze;
- **GAN** per generare dati nuovi;
- **transformer** e da li' i **grandi modelli linguistici (LLM)**.

un unico continuum, dai minimi quadrati di stamattina fino agli LLM. fine del corso, e grazie.